In [ ]:
import json
import os
import re
import time
from pathlib import Path
from typing import List, Dict, Any, Optional
import google.generativeai as genai



API_KEY = ""  # 🔑 Thay bằng API key thực tế

FILE_LIST = [
    "/kaggle/input/15filehistxt/Lch s Vit Nam tp 13 T nm 1965 n nm 1975-Nguyn Vn Nht-2017.txt", 
    "/kaggle/input/15filehistxt/Lch s Vit Nam tp 14 T nm 1975 n nm 1986-Trn c Cng-2017.txt", 
    "/kaggle/input/15filehistxt/Lch s Vit Nam tp 15 T nm 1986 n nm 2000-Nguyn Ngc Mo-2017.txt"
]  # 📁 Danh sách file .txt session này sẽ xử lý

OUTPUT_FILE = "/kaggle/working/rag_data_session_5.jsonl"  # 💾 File output JSONL

INPUT_DIR = "/kaggle/input/15filehistxt"  # 📂 Thư mục chứa file .txt

MODEL_PRIORITY_LIST = [
    'gemini-2.5-flash-lite',
    'gemini-2.5-flash', 
    'gemini-2.5-pro',
    'gemini-2.0-flash',
    'gemini-2.0-flash-lite',
]

BATCH_CHAR_LIMIT = 10000  # ⚡ Giới hạn ký tự mỗi batch
REQUESTS_BEFORE_BREAK = 5  # ⏸️ Số request trước khi nghỉ
BREAK_DURATION = 10  # 💤 Thời gian nghỉ (giây)

# ================================
# KHÔNG CHỈNH SỬA PHẦN DƯỚI ĐÂY
# ================================
class TextProcessor:
    def __init__(self):
        self.request_count = 0
        genai.configure(api_key=API_KEY)
    
    def check_and_break(self):
        """Kiểm tra và nghỉ nếu đã đủ số request"""
        self.request_count += 1
        if self.request_count % REQUESTS_BEFORE_BREAK == 0:
            print(f"⏸️ Đã thực hiện {self.request_count} requests, nghỉ {BREAK_DURATION}s...")
            time.sleep(BREAK_DURATION)

def create_text_batches(file_path: str, char_limit: int = BATCH_CHAR_LIMIT) -> List[str]:
    """
    Đọc file và chia thành các batch dựa trên giới hạn ký tự
    Ưu tiên tách theo đoạn văn (\n\n)
    """
    try:
        # Thử các encoding khác nhau
        encodings = ['utf-8', 'cp1252', 'latin-1', 'utf-16']
        content = None
        
        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    content = f.read()
                break
            except UnicodeDecodeError:
                continue
        
        # Fallback với ignore errors
        if content is None:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
        
        if not content.strip():
            print(f"⚠️ File rỗng: {file_path}")
            return []
        
        # Tách thành các đoạn văn
        paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
        
        batches = []
        current_batch = ""
        current_size = 0
        
        for paragraph in paragraphs:
            para_size = len(paragraph)
            
            # Nếu đoạn văn quá lớn, tách nhỏ hơn
            if para_size > char_limit:
                if current_batch:
                    batches.append(current_batch)
                    current_batch = ""
                    current_size = 0
                
                # Chia đoạn lớn thành các phần nhỏ
                parts = [paragraph[i:i+char_limit] for i in range(0, len(paragraph), char_limit)]
                batches.extend(parts[:-1])  # Thêm tất cả trừ phần cuối
                current_batch = parts[-1] if parts else ""
                current_size = len(current_batch)
                continue
            
            # Nếu thêm đoạn này vượt quá giới hạn, lưu batch hiện tại
            if current_size + para_size > char_limit and current_batch:
                batches.append(current_batch)
                current_batch = paragraph
                current_size = para_size
            else:
                # Thêm đoạn vào batch hiện tại
                if current_batch:
                    current_batch += "\n\n" + paragraph
                    current_size += para_size + 2  # +2 cho \n\n
                else:
                    current_batch = paragraph
                    current_size = para_size
        
        # Thêm batch cuối cùng nếu còn
        if current_batch:
            batches.append(current_batch)
        
        print(f"📦 Đã tạo {len(batches)} batch từ file {os.path.basename(file_path)}")
        
        # In thông tin về batch đầu tiên để kiểm tra
        if batches:
            first_batch_preview = batches[0][:500] + "..." if len(batches[0]) > 500 else batches[0]
            print(f"   👀 Batch 1 preview ({len(batches[0])} chars): {first_batch_preview}")
        
        return batches
        
    except Exception as e:
        print(f"❌ Lỗi đọc file {file_path}: {str(e)}")
        return []

def get_rag_prompt(text_batch: str) -> str:
    """Tạo prompt cho việc chuyển đổi sang RAG format"""
    return f"""
Bạn là một chuyên gia xử lý dữ liệu Lịch sử Việt Nam. Nhiệm vụ của bạn là chuyển đổi văn bản thô sau đây thành một DANH SÁCH (LIST) các đối tượng JSON (DICTIONARY) cho hệ thống RAG.

YÊU CẦU XỬ LÝ:
1. Đọc kỹ văn bản thô.
2. Tách văn bản thành các đoạn (chunks) hợp lý, mỗi đoạn tập trung vào MỘT sự kiện hoặc chủ đề.
3. VIẾT LẠI (Rewrite) nội dung mỗi đoạn (trường "text") sao cho nó có thể đứng độc lập, đầy đủ ngữ cảnh (thay thế đại từ "ông", "năm đó" bằng tên/năm cụ thể).
4. Trích xuất Metadata chính xác cho mỗi đoạn.

VĂN BẢN GỐC:
---
{text_batch}
---

CẤU TRÚC JSON OUTPUT BẮT BUỘC (Trả về một List[Dict[str, Any]]):
[
  {{
    "text": "Nội dung đoạn 1 đã được viết lại, đầy đủ ngữ cảnh...",
    "metadata": {{
      "nam": <Số nguyên hoặc null>,
      "trieu_dai": "<Tên triều đại hoặc giai đoạn>",
      "thuc_the": ["<Tên người>", "<Địa danh>"],
      "chu_de": "<Tóm tắt 3-5 từ>"
    }}
  }}
]

QUAN TRỌNG:
- Luôn trả về một LIST, ngay cả khi chỉ có 1 đoạn.
- "nam" có thể là null nếu không xác định được năm.
- "thuc_the" là list chứa tên người và địa danh quan trọng.
- "chu_de" phải ngắn gọn, 3-5 từ mô tả chủ đề.
- Đảm bảo mỗi đoạn "text" có thể đứng độc lập, không phụ thuộc vào ngữ cảnh bên ngoài.
"""

def process_batch_with_fallback(processor: TextProcessor, text_batch: str, batch_index: int, file_name: str) -> Optional[List[Dict]]:
    """Xử lý một batch với cơ chế fallback model"""
    
    for model_index, model_name in enumerate(MODEL_PRIORITY_LIST):
        try:
            processor.check_and_break()  # Kiểm tra và nghỉ nếu cần
            
            print(f"🔄 [{file_name}] Batch {batch_index} - Thử model: {model_name}")
            
            model = genai.GenerativeModel(model_name)
            prompt = get_rag_prompt(text_batch)
            
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.1,
                    max_output_tokens=8192,
                    top_p=0.8,
                    response_mime_type="application/json"
                )
            )
            
            # Parse JSON response
            result_text = response.text.strip()
            
            # Loại bỏ markdown code block nếu có
            if result_text.startswith('```json'):
                result_text = result_text[7:]
            if result_text.endswith('```'):
                result_text = result_text[:-3]
            
            result_data = json.loads(result_text)
            
            # Kiểm tra cấu trúc kết quả
            if not isinstance(result_data, list):
                raise ValueError("Kết quả không phải là list")
            
            # Kiểm tra từng item trong list
            for item in result_data:
                if not isinstance(item, dict):
                    raise ValueError("Item không phải là dictionary")
                if "text" not in item or "metadata" not in item:
                    raise ValueError("Thiếu trường bắt buộc 'text' hoặc 'metadata'")
                if not isinstance(item["metadata"], dict):
                    raise ValueError("metadata không phải là dictionary")
                
                # Validate metadata fields
                metadata = item["metadata"]
                if "nam" in metadata and metadata["nam"] is not None:
                    if not isinstance(metadata["nam"], int):
                        # Thử convert sang int nếu có thể
                        try:
                            metadata["nam"] = int(metadata["nam"])
                        except (ValueError, TypeError):
                            metadata["nam"] = None
                
                if "trieu_dai" not in metadata:
                    metadata["trieu_dai"] = ""
                if "thuc_the" not in metadata:
                    metadata["thuc_the"] = []
                if "chu_de" not in metadata:
                    metadata["chu_de"] = ""
            
            print(f"✅ [{file_name}] Batch {batch_index} - Thành công với {model_name}: {len(result_data)} đoạn")
            
            # 🔍 IN RA MẪU DỮ LIỆU ĐỂ KIỂM TRA
            if result_data:
                print(f"   📋 MẪU DỮ LIỆU BATCH {batch_index} (đoạn đầu tiên):")
                first_item = result_data[0]
                sample_text = first_item['text'][:300] + "..." if len(first_item['text']) > 300 else first_item['text']
                print(f"      📝 Text: {sample_text}")
                print(f"      🏷️  Metadata: {json.dumps(first_item['metadata'], ensure_ascii=False, indent=8)}")
                
                # Nếu có nhiều hơn 1 đoạn, in thông tin tổng quan
                if len(result_data) > 1:
                    print(f"      📊 Các chủ đề khác trong batch: {[item['metadata'].get('chu_de', '') for item in result_data[1:4]]}")
            
            return result_data
            
        except Exception as e:
            error_msg = str(e)[:200]
            print(f"❌ [{file_name}] Batch {batch_index} - Lỗi với {model_name}: {error_msg}")
            
            # Nghỉ ngắn trước khi thử model tiếp theo
            time.sleep(2)
            continue
    
    # Đã thử tất cả model mà vẫn lỗi
    print(f"🚨 [{file_name}] Batch {batch_index} - Đã thử tất cả model, bỏ qua batch này")
    return None

def main():
    """Hàm chính xử lý tất cả file trong FILE_LIST"""
    
    # Khởi tạo processor
    processor = TextProcessor()
    
    # Đảm bảo thư mục output tồn tại
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    print(f"🚀 BẮT ĐẦU XỬ LÝ RAG DATA")
    print(f"📁 Input directory: {INPUT_DIR}")
    print(f"💾 Output file: {OUTPUT_FILE}")
    print(f"📄 Files to process: {FILE_LIST}")
    print(f"🤖 Models: {MODEL_PRIORITY_LIST}")
    print("=" * 60)
    
    total_processed = 0
    total_failed = 0
    
    # Mở file output một lần duy nhất
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as output_f:
        for file_name in FILE_LIST:
            file_path = os.path.join(INPUT_DIR, file_name)
            
            if not os.path.exists(file_path):
                print(f"❌ File không tồn tại: {file_path}")
                total_failed += 1
                continue
            
            print(f"\n📖 Đang xử lý: {file_name}")
            
            # Tạo batches từ file
            batches = create_text_batches(file_path)
            
            if not batches:
                print(f"⚠️ Không có batch nào từ file: {file_name}")
                continue
            
            file_processed = 0
            file_failed = 0
            
            for batch_index, batch_content in enumerate(batches):
                # Xử lý batch với fallback
                result = process_batch_with_fallback(processor, batch_content, batch_index + 1, file_name)
                
                if result:
                    # Ghi từng đoạn vào file JSONL
                    for item in result:
                        # Thêm source file vào metadata
                        item["metadata"]["source_file"] = file_name
                        
                        # Ghi dòng JSON
                        json_line = json.dumps(item, ensure_ascii=False)
                        output_f.write(json_line + '\n')
                        output_f.flush()  # Đảm bảo ghi ngay lập tức
                    
                    file_processed += len(result)
                    total_processed += len(result)
                else:
                    file_failed += 1
                    total_failed += 1
            
            print(f"📊 Kết quả {file_name}: {file_processed} đoạn thành công, {file_failed} batch thất bại")
    
    print(f"\n{'='*60}")
    print(f"🎉 HOÀN THÀNH XỬ LÝ RAG DATA")
    print(f"📈 Tổng số đoạn đã xử lý: {total_processed}")
    print(f"📉 Tổng số batch thất bại: {total_failed}")
    print(f"🔢 Tổng số request: {processor.request_count}")
    print(f"💾 Kết quả được lưu tại: {OUTPUT_FILE}")
    
    # 🔍 IN RA THỐNG KÊ CUỐI CÙNG VỀ DỮ LIỆU ĐÃ XỬ LÝ
    print(f"\n📊 THỐNG KÊ DỮ LIỆU ĐÃ XỬ LÝ:")
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if lines:
                print(f"   📁 Tổng số dòng trong file output: {len(lines)}")
                
                # Phân tích 3 dòng đầu tiên để hiển thị mẫu
                print(f"   👀 Mẫu dữ liệu đầu ra (3 dòng đầu):")
                for i, line in enumerate(lines[:3]):
                    data = json.loads(line.strip())
                    text_preview = data['text'][:150] + "..." if len(data['text']) > 150 else data['text']
                    print(f"      Dòng {i+1}:")
                    print(f"        Text: {text_preview}")
                    print(f"        Metadata: {data['metadata']}")
            else:
                print("   ⚠️ File output rỗng")
    except Exception as e:
        print(f"   ❌ Không thể đọc file output để thống kê: {e}")

if __name__ == "__main__":
    main()